# CareerNet — search the advice, then ask it questions

CareerNet is a collection of 6,000 real career questions asked by young people on
CareerVillage.org, and the 16,130 answers volunteers gave them. Every question and answer
has been read and labelled by trained human annotators: which occupation it is about, what
the asker was trying to achieve, and how correct, complete and clear the answer is. Each
answer also has a label for the kind of reasoning it uses, assigned by a language model
trained on examples labelled by hand.

This notebook lets you do two things with that collection:

1. **Search it by meaning.** Type a question in your own words and see the closest
   questions or answers in CareerNet, along with charts of what the results are about.
2. **Chat with it.** Ask a career question and get an answer written from real
   CareerNet advice, with the source material shown alongside so you can judge it.

**How to use this notebook.** Work down the page and press the ▶ button on the left of
each grey cell, in order. The code is hidden; you only need the play buttons. Where a step
takes more than a few seconds, the text says so. Nothing here needs an account, a
password or a licence.

**Before you start:** in the menu, choose *Runtime → Change runtime type* and pick
**T4 GPU**. The chat section at the end needs it; the rest would run without it.


## 1. Install the tools

⏳ **This step takes a while** — about two to three minutes. A lot of text scrolls past; that is normal. It is
finished when the play button stops spinning.


In [ ]:
#@title Install (2-3 minutes)
!pip -q install -U onnxruntime 'sentence-transformers[onnx]>=5.0' 'transformers>=4.56' plotly accelerate ipywidgets

import torch, numpy as np
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'Ready. Graphics card: {p.name} ({p.total_memory/1e9:.0f} GB)')
else:
    print('Ready, but no graphics card was found. Sections 1-4 will work; section 5 (chat) '
          'will be very slow or fail. Use Runtime -> Change runtime type -> T4 GPU.')


## 2. Connect to the data

The collection lives in six files in the `Notebooks/data` folder of the
[CareerNet GitHub repository](https://github.com/RenaissancePhilanthropy/careernet-data/tree/main/Notebooks/data).
This step checks that the notebook can reach them. Nothing to sign in to.

If you are running this notebook from your own copy of the repository, it uses the local
`data` folder next to the notebook instead of downloading.

In [ ]:
#@title Where the data is
# The six CareerNet data files. By default they are read from the GitHub repository.
APP_BASE  = 'https://raw.githubusercontent.com/RenaissancePhilanthropy/careernet-data/main/Notebooks/data/'

# Advanced, normally left alone: read the files from a folder on this machine instead.
# Left as None, a 'data' folder next to this notebook is used if one exists.
LOCAL_DIR = None    # e.g. '/content/data'

import os, urllib.request
DATA_FILES = ('manifest.json', 'dense_head.bin', 'questions.meta.json',
              'questions.vec.bin', 'answers.meta.json', 'answers.vec.bin')
if LOCAL_DIR is None and all(os.path.exists(os.path.join('data', f)) for f in DATA_FILES):
    LOCAL_DIR = 'data'

if LOCAL_DIR:
    missing = [f for f in DATA_FILES if not os.path.exists(os.path.join(LOCAL_DIR, f))]
    if missing:
        raise SystemExit(f'Could not find {missing} in {LOCAL_DIR}. Check that LOCAL_DIR is correct.')
    print(f'Using the data files in {LOCAL_DIR}.')
else:
    try:
        urllib.request.urlopen(APP_BASE + 'manifest.json').read()
    except Exception as e:
        raise SystemExit(f'Could not reach the data files at {APP_BASE} ({e}). Check your internet connection.')
    print('Connected. The data files will be read from GitHub.')

## 3. Load the collection

This reads the questions, the answers and their labels into memory — about 45 MB,
usually under a minute to download.

It also loads something that needs a short explanation. To search by *meaning* rather than
by matching words, every question and answer has already been turned into a **meaning
fingerprint**: a list of 768 numbers that captures what the text is about, so that two
pieces of text about the same thing have similar fingerprints even if they use different
words. Those fingerprints were computed in advance and are included in the data files.
Later, your search will be turned into the same kind of fingerprint and compared against
them.


In [ ]:
#@title Load questions, answers and labels
import json, urllib.request

def fetch(name, binary=False):
    if LOCAL_DIR:
        mode = 'rb' if binary else 'r'
        kw = {} if binary else {'encoding': 'utf-8'}
        with open(f'{LOCAL_DIR}/{name}', mode, **kw) as fh:
            return fh.read()
    if not APP_BASE:
        raise RuntimeError('Point APP_BASE or LOCAL_DIR at the data files.')
    raw = urllib.request.urlopen(APP_BASE + name).read()
    return raw if binary else raw.decode('utf-8')

MAN = json.loads(fetch('manifest.json'))
SOC_TITLE   = MAN['soc_detail']     # occupation code -> occupation name
MAJOR_TITLE = MAN['soc_major']      # occupation family code -> family name
GOAL_LABEL  = MAN['goal_labels']    # goal code -> plain-English goal
SOC_LMI     = MAN['soc_lmi']        # occupation -> government wage and employment figures

# The fingerprints were made by a model with two final layers that the public copy of the
# model leaves out. This matrix stands in for them so our fingerprints match the stored ones.
HEAD = np.frombuffer(fetch('dense_head.bin', binary=True), dtype=np.float32).reshape(768, 768)

def load(name):
    spec = MAN['corpora'][name]
    meta = json.loads(fetch(f'{name}.meta.json'))
    vec = np.frombuffer(fetch(f'{name}.vec.bin', binary=True), dtype=np.int8)
    vec = vec.reshape(spec['n'], spec['dim']).astype(np.float32) * spec['scale']
    recs = []
    for m in meta:
        recs.append(dict(text=m['t'], title=m.get('ti', ''), domain=m['d'],
                         soc=m['soc'], major=m['maj'], scenario=m['sc'], goals=m['g'],
                         flags=m['fl'], year=m['y'], when=m['w'],
                         reasoning=m.get('rl'), question_title=m.get('qt'),
                         quality=m.get('ql'), answer_idx=m.get('ai'), q_idx=m.get('qi')))
    return recs, vec

QUESTIONS, Q_EMB = load('questions')
ANSWERS,   A_EMB = load('answers')
CORPORA = {'questions': (QUESTIONS, Q_EMB), 'answers': (ANSWERS, A_EMB)}
print(f'Loaded {len(QUESTIONS):,} questions and {len(ANSWERS):,} answers, '
      f'covering {len(SOC_TITLE)} occupations.')


### The search model

To turn *your* words into a fingerprint, the notebook needs the same small language model
that produced the stored ones. It downloads once (about 300 MB) and then runs on the
ordinary processor — a search takes a fraction of a second.

⏳ **This step takes a while** — one to two minutes for the download.

> You may see a message saying *"No sentence-transformers model found ... Creating a new one
> with mean pooling"*. That is expected and not a problem.

The last line of output is a self-check: the notebook re-fingerprints one stored answer and
confirms it gets the same result. If that check fails, stop and let us know.


In [ ]:
#@title Download the search model (1-2 minutes)
from sentence_transformers import SentenceTransformer

ONNX_REPO = 'onnx-community/embeddinggemma-300m-ONNX'
ONNX_FILE = 'onnx/model_quantized.onnx'
QUERY_PROMPT = 'task: search result | query: '
DOC_PROMPT   = 'title: none | text: '

encoder = SentenceTransformer(ONNX_REPO, backend='onnx',
                              model_kwargs={'file_name': ONNX_FILE}, device='cpu')
encoder.max_seq_length = 512

def fingerprint(texts, prompt):
    v = encoder.encode([prompt + t for t in texts], normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=False)
    v = v @ HEAD.T
    return v / np.linalg.norm(v, axis=1, keepdims=True)

def fingerprint_query(text):
    return fingerprint([text], QUERY_PROMPT)[0]

# Self-check: re-fingerprint one stored answer and compare with the stored fingerprint.
# Agreement close to 1.0 means everything lines up.
probe = 3
check = float(fingerprint([ANSWERS[probe]['text']], DOC_PROMPT)[0] @ A_EMB[probe])
if check > 0.9:
    print(f'Self-check passed (agreement {check:.3f} out of 1.0). Ready to search.')
else:
    raise SystemExit(f'Self-check FAILED (agreement {check:.3f}). Searches would be wrong; please report this.')


## 4. Search by meaning

Run the cell below and a search box appears. Type a question the way a student might ask
it — you do not need the right keywords, because the search compares meaning, not words.
It returns the volunteer answers closest in meaning to what you typed.

Each result shows the question it was replying to, the answer, the occupation it is about,
what the asker was trying to do, and the kind of reasoning the volunteer used. The occupation
and the asker's goal were assigned by human annotators. The reasoning label was assigned
by a language model trained on hand-labelled examples.

You can narrow to one subject area and choose how many results to see. Or press one of the
example buttons:

- *How do I become a registered nurse, and how long does it take?*
- *What does a software developer actually do day to day?*
- *How do I get an internship when I have no experience?*
- *Is cybersecurity a good career to get into?*


In [ ]:
#@title Search tools (run once)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
import re
import ipywidgets as W
from IPython.display import display, HTML

DOMAIN_NAME = {'general': 'General career advice', 'health': 'Health professions',
               'technology': 'Technology professions'}

def search(query, k=20, domain=None):
    qv = fingerprint_query(query)
    keep = np.array([domain is None or r['domain'] == domain for r in ANSWERS])
    sims = np.where(keep, A_EMB @ qv, -np.inf)
    k = min(k, int(keep.sum()))
    if k == 0:
        return []
    idx = np.argpartition(-sims, k - 1)[:k]
    idx = idx[np.argsort(-sims[idx])]
    return [(float(sims[i]), ANSWERS[i]) for i in idx if np.isfinite(sims[i])]

def clean(text):
    """A few answers still carry web formatting tags; drop them for display."""
    return ' '.join(re.sub(r'<[^>]+>', ' ', text).split())

def occupation_of(r):
    occ = '; '.join(SOC_TITLE.get(c, c) for c in r['soc'])
    if occ:
        return occ
    return r['flags'][0] if r['flags'] else 'no specific occupation'

def show_hits(hits, chars=300):
    for rank, (s, r) in enumerate(hits, 1):
        print()
        print(f"{rank}. ({DOMAIN_NAME[r['domain']]}, {r['when']})")
        if r.get('question_title'):
            print('   asked:', r['question_title'][:110])
        body = clean(r['text'])
        print('   answer:', body[:chars] + ('...' if len(body) > chars else ''))
        print('   occupation:', occupation_of(r))
        print('   asker wanted to:', '; '.join(GOAL_LABEL.get(g, g) for g in r['goals']) or 'unclear')
        if r.get('reasoning'):
            print('   reasoning used:', '; '.join(r['reasoning']))

print('Search tools ready. Run the next cell to open the search box.')


In [ ]:
#@title Open the search box
LAST = {'query': None, 'hits': []}    # remembered for the charts cell below

q_box = W.Text(placeholder='Ask in your own words, then press Enter or Search',
               layout=W.Layout(width='70%'))
go_btn = W.Button(description='Search', button_style='primary')
domain_dd = W.Dropdown(description='Area', options=[('All areas', None)] +
    [(v, k) for k, v in DOMAIN_NAME.items()])
n_dd = W.Dropdown(description='Results', options=[5, 10, 20], value=10)
out = W.Output()

def run_search(_=None):
    query = q_box.value.strip()
    if not query:
        return
    out.clear_output()
    go_btn.disabled = True
    try:
        with out:
            hits = search(query, k=n_dd.value, domain=domain_dd.value)
            LAST['query'], LAST['hits'] = query, hits
            if not hits:
                print('Nothing found. Try a different area.')
                return
            print(f'The {len(hits)} answers closest in meaning to: "{query}"')
            show_hits(hits)
    finally:
        go_btn.disabled = False

def example_button(text):
    b = W.Button(description=text, layout=W.Layout(width='auto'), tooltip=text)
    def click(_):
        q_box.value = text
        run_search()
    b.on_click(click)
    return b

go_btn.on_click(run_search)
if hasattr(q_box, 'on_submit'):
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        q_box.on_submit(run_search)

examples = ['How do I become a registered nurse, and how long does it take?', 'What does a software developer actually do day to day?', 'How do I get an internship when I have no experience?', 'Is cybersecurity a good career to get into?']
display(W.VBox([W.HBox([q_box, go_btn]), W.HBox([domain_dd, n_dd]),
                W.HTML('<span style="color:#666">Try one of these:</span>'),
                W.HBox([example_button(e) for e in examples], layout=W.Layout(flex_flow='row wrap')),
                out]))


### What your last search is made of

Run this cell after a search to get three charts about the results it returned. Run it
again after each new search.

- **What the results are about** — the occupation families, asker goals, and kinds of
  reasoning among the answers.
- **When they were asked** — the collection runs from 2011 to 2025. Try a search about AI
  and one about résumés and compare.
- **The occupations in the labour market** — the occupations in your results, with average
  pay and number of workers from the U.S. Bureau of Labor Statistics.


In [ ]:
#@title Charts for the last search
TEAL, BLUE, DARK = '#1EA69A', '#0099E8', '#01579B'
LAYOUT = dict(font=dict(family='Inter, sans-serif', size=12), plot_bgcolor='rgba(0,0,0,0)',
              paper_bgcolor='white')
GRID = 'rgba(148,163,184,.25)'

def compose_chart(hits):
    panels = [
        ('Occupation family', Counter(MAJOR_TITLE.get(m, m) for _, r in hits for m in r['major']), TEAL),
        ('What the asker wanted', Counter(GOAL_LABEL.get(g, g) for _, r in hits for g in r['goals']), BLUE),
        ('Kind of reasoning', Counter(v for _, r in hits for v in (r['reasoning'] or [])), DARK),
    ]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[p[0] for p in panels],
                        horizontal_spacing=0.30)
    for col, (title, counts, color) in enumerate(panels, 1):
        top = counts.most_common(8)[::-1]
        if not top:
            continue
        labels = [t[0][:32] + ('...' if len(t[0]) > 32 else '') for t in top]
        fig.add_trace(go.Bar(y=labels, x=[t[1] for t in top], orientation='h',
                             marker_color=color, showlegend=False,
                             hovertemplate='%{y}<br>%{x} of ' + str(len(hits)) +
                                           ' results<extra></extra>'),
                      row=1, col=col)
    fig.update_layout(height=340, margin=dict(t=70, b=45, l=10, r=20),
                      title=f'What your {len(hits)} results are about', **LAYOUT)
    fig.update_xaxes(dtick=1, gridcolor=GRID)
    fig.update_yaxes(tickfont=dict(size=10))
    return fig

def year_chart(hits):
    counts = Counter(r['year'] for _, r in hits if r['year'])
    if not counts:
        return None
    years = list(range(min(counts), max(counts) + 1))
    fig = go.Figure(go.Bar(x=years, y=[counts.get(y, 0) for y in years], marker_color=BLUE,
                           hovertemplate='%{x}: %{y} results<extra></extra>'))
    fig.update_layout(height=260, margin=dict(t=60, b=45, l=10, r=20),
                      title=f'When these {len(hits)} were asked', **LAYOUT)
    fig.update_yaxes(gridcolor=GRID)
    return fig

def lmi_chart(hits, top=12):
    agg = {}
    for _, r in hits:
        for c in r['soc']:
            if c in SOC_LMI:
                agg.setdefault(c, dict(SOC_LMI[c], hits=0))['hits'] += 1
    rows = sorted(agg.values(), key=lambda d: (-d['hits'], -d['wage']))[:top][::-1]
    if not rows:
        return None
    fig = go.Figure(go.Bar(
        y=[r['title'][:40] for r in rows], x=[r['wage'] for r in rows], orientation='h',
        marker=dict(color=[r['employment'] for r in rows],
                    colorscale=[[0, '#E3F5FD'], [1, BLUE]],
                    colorbar=dict(title='Workers', thickness=12, len=0.6)),
        customdata=[[r['employment'], r['hits']] for r in rows],
        hovertemplate='<b>%{y}</b><br>Average pay: $%{x:,.0f} a year'
                      '<br>Workers in the U.S.: %{customdata[0]:,.0f}'
                      '<br>In your results: %{customdata[1]}<extra></extra>'))
    fig.update_layout(height=460, margin=dict(t=70, b=45, l=250, r=20),
                      title='Occupations in your results: average pay (bar) and number of workers (colour)',
                      xaxis_title='Average annual pay', **LAYOUT)
    fig.update_xaxes(tickprefix='$', tickformat=',.0f', gridcolor=GRID)
    return fig

if not LAST['hits']:
    print('Run a search first, then run this cell.')
else:
    print(f'Charts for: "{LAST["query"]}" ({len(LAST["hits"])} results)')
    compose_chart(LAST['hits']).show()
    f = year_chart(LAST['hits'])
    if f: f.show()
    f = lmi_chart(LAST['hits'])
    if f:
        f.show()
    else:
        print('None of these results names an occupation with government pay figures.')


## 5. Chat with the collection

This is the part a web page cannot do. When you ask a question, the notebook first finds
the five CareerNet answers closest in meaning to it, then hands those answers to a
language model with the instruction: *answer this question using only this material, and
say so if the material does not cover it*. The model writes a reply in plain language.

The answers it drew on are listed under each reply on purpose. The point is to see what the
model was working from, not to take its word for it.

The chat remembers the conversation, so follow-ups like *"tell me more about the second
one"* work. Press **Start over** to clear it. Each reply takes ten to twenty seconds; a
spinner shows while it is working.

⏳ **This step takes a while** — the language model is about 8 GB and downloads once, roughly five to ten
minutes.

Good questions to start with — the collection has plenty of well-rated advice on each:

- *I want to become a registered nurse. What should I be doing in high school and college?*
- *How do I get my first internship when I have no experience?*
- *What is the difference between a software developer and a data scientist?*
- *How should I prepare for a job interview?*

> **A note on the answers.** They are generated by a general-purpose model from a handful of
> volunteer answers, and nothing here checks whether they are accurate or good advice.
> Treat them as a demonstration of what the labelled collection makes possible, not as
> career guidance.


In [ ]:
#@title Download the chat model (5-10 minutes)
from transformers import AutoModelForCausalLM, AutoTokenizer

# Qwen3-4B fits a Colab T4. On a card with less than 12 GB, use 'Qwen/Qwen3-1.7B' instead.
GEN_ID = 'Qwen/Qwen3-4B'

gen_tok = AutoTokenizer.from_pretrained(GEN_ID)
gen = AutoModelForCausalLM.from_pretrained(
    GEN_ID,
    dtype=(torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16),
    device_map='auto')
gen.eval()
print('Chat model ready.')


In [ ]:
#@title Chat tools (run once)
import threading, time

SYSTEM = ('You are helping a young person with a career question. Use only the advice in the '
          'SOURCE MATERIAL, which comes from real answers given by volunteers on CareerVillage. '
          'If the material does not cover something, say so rather than inventing it. '
          'Write four to six sentences in plain language.')

MAX_PROMPT_TOKENS = 6000   # how much conversation plus source material the model is given
MAX_REPLY_TOKENS  = 400
PASSAGE_CHARS     = [1200, 700, 350]   # how much of each source answer to include; shrinks if needed

def retrieve(question, k=5):
    sims = A_EMB @ fingerprint_query(question)
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

def source_block(idx, chars):
    blocks = []
    for n, (i, _) in enumerate(idx, 1):
        r = ANSWERS[i]
        body = clean(r['text'])[:chars]
        blocks.append(f'[{n}] (about: {occupation_of(r)})' + chr(10) + body)
    return 'SOURCE MATERIAL:' + chr(10) * 2 + (chr(10) * 2).join(blocks)

def count_tokens(msgs):
    return len(gen_tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                           enable_thinking=False))

def fit_prompt(question, idx, history):
    """Build the messages, dropping the oldest exchanges and then shortening the source
    material until everything fits the token budget. Returns (msgs, note) or (None, reason)."""
    history = list(history)
    note = ''
    for chars in PASSAGE_CHARS:
        user = source_block(idx, chars) + chr(10) * 2 + 'QUESTION: ' + question
        while True:
            msgs = [{'role': 'system', 'content': SYSTEM}] + history + [{'role': 'user', 'content': user}]
            if count_tokens(msgs) <= MAX_PROMPT_TOKENS:
                return msgs, note
            if not history:
                break
            history = history[2:]
            note = '(The conversation got long, so the earliest exchanges were forgotten.)'
    return None, 'That question is too long for the model to work with. Please shorten it.'

def generate(msgs):
    text = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    enc = gen_tok(text, return_tensors='pt').to(gen.device)
    with torch.no_grad():
        out = gen.generate(**enc, max_new_tokens=MAX_REPLY_TOKENS, do_sample=True,
                           temperature=0.7, top_p=0.9, pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

class Chat:
    def __init__(self):
        self.history = []   # alternating user / assistant turns, questions stored without source material

    def ask(self, question):
        # For a short follow-up, search using the previous question too, so 'tell me more'
        # still finds material on the same topic.
        lookup = question
        if self.history and len(question.split()) < 6:
            lookup = self.history[-2]['content'] + ' ' + question
        idx = retrieve(lookup)
        msgs, note = fit_prompt(question, idx, self.history)
        if msgs is None:
            return note, [], ''
        try:
            answer = generate(msgs)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            self.history = []
            return ('The graphics card ran out of memory on that one. The conversation has been '
                    'cleared; please ask again, perhaps more briefly.'), [], ''
        self.history += [{'role': 'user', 'content': question},
                         {'role': 'assistant', 'content': answer}]
        self.history = self.history[-20:]   # keep the last ten exchanges at most
        return answer, idx, note

SPINNER_CSS = ('<style>@keyframes cn-spin{to{transform:rotate(360deg)}}'
               '.cn-spin{display:inline-block;width:14px;height:14px;border:2px solid #cbd5e1;'
               'border-top-color:#0099E8;border-radius:50%;animation:cn-spin .8s linear infinite;'
               'vertical-align:middle;margin-right:8px}</style>')

class Spinner:
    """Shows a spinning circle and a running clock in `widget` until stopped."""
    def __init__(self, widget):
        self.widget = widget
        self._stop = threading.Event()
    def __enter__(self):
        self._stop.clear()
        t0 = time.time()
        def tick():
            while not self._stop.is_set():
                self.widget.value = (SPINNER_CSS + '<span class="cn-spin"></span>'
                                     f'<span style="color:#666">Thinking... {time.time() - t0:.0f}s '
                                     '(usually 10-20 seconds)</span>')
                self._stop.wait(1)
        threading.Thread(target=tick, daemon=True).start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
        self.widget.value = ''

print('Chat tools ready. Run the next cell to open the chat box.')


In [ ]:
#@title Open the chat box
chat = Chat()
c_box = W.Text(placeholder='Ask a career question, then press Enter or Ask',
               layout=W.Layout(width='70%'))
ask_btn = W.Button(description='Ask', button_style='primary')
reset_btn = W.Button(description='Start over')
show_src = W.Checkbox(value=True, description='Show the advice it drew on', indent=False)
status = W.HTML()
log = W.Output()

def esc(s):
    return s.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')

def on_ask(_=None):
    q = c_box.value.strip()
    if not q:
        return
    c_box.value = ''
    ask_btn.disabled = True
    ask_btn.description = 'Thinking...'
    with log:
        display(HTML(f'<p style="margin:14px 0 4px"><b>You:</b> {esc(q)}</p>'))
    try:
        with Spinner(status):
            answer, idx, note = chat.ask(q)
    finally:
        ask_btn.disabled = False
        ask_btn.description = 'Ask'
    with log:
        display(HTML(f'<p style="margin:4px 0"><b>CareerNet:</b> {esc(answer)}</p>'))
        if note:
            display(HTML(f'<p style="color:#888;margin:0"><i>{esc(note)}</i></p>'))
        if idx and show_src.value:
            rows = []
            for n, (i, s) in enumerate(idx, 1):
                r = ANSWERS[i]
                q_ = r['quality'] or [None, None, None]
                rated = f' &middot; rated {q_[0]}/4 for correctness' if q_[0] else ''
                snippet = esc(clean(r['text'])[:220])
                rows.append(f'<li><b>{esc(occupation_of(r))}</b>{rated}<br>'
                            f'<span style="color:#555">{snippet}...</span></li>')
            display(HTML('<details style="margin:4px 0 0 12px"><summary style="color:#666;cursor:pointer">'
                         'The five CareerNet answers this drew on</summary><ol style="font-size:90%">'
                         + ''.join(rows) + '</ol></details>'))

def on_reset(_=None):
    chat.history = []
    log.clear_output()
    with log:
        print('Conversation cleared.')

def example_button(text):
    b = W.Button(description=text, layout=W.Layout(width='auto'), tooltip=text)
    def click(_):
        c_box.value = text
        on_ask()
    b.on_click(click)
    return b

ask_btn.on_click(on_ask)
reset_btn.on_click(on_reset)
if hasattr(c_box, 'on_submit'):
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        c_box.on_submit(on_ask)

examples = ['I want to become a registered nurse. What should I be doing in high school and college?', 'How do I get my first internship when I have no experience?', 'What is the difference between a software developer and a data scientist?', 'How should I prepare for a job interview?']
display(W.VBox([W.HBox([c_box, ask_btn, reset_btn]), show_src,
                W.HTML('<span style="color:#666">Try one of these:</span>'),
                W.HBox([example_button(e) for e in examples], layout=W.Layout(flex_flow='row wrap')),
                status, log]))


## What you just used

Everything above rests on the labels in CareerNet. Most come from human annotators reading
each question and answer and recording what it is about, what the asker wanted, and how
good the answer is; the occupation-to-pay link is built on those occupation labels. The
reasoning labels were assigned by a language model trained on hand-labelled examples.
The language models in this notebook only do two jobs: turn text into meaning fingerprints so it can be
searched, and write a reply from the material the search finds.
